<h1>🧬 Biofilter — Report: <code>annotate_variant</code></h1>

What the bundle knows about a list of variants: identity, gnomAD joint
frequencies, in-silico predictions, and one row per transcript the
variant was annotated against.

### 1. Open a bundle

In [ ]:
from pathlib import Path

from biofilter import Biofilter

# Leave as None to use `[database] bundle` from .biofilter.toml.
BUNDLE = None
REPORT = "annotate_variant"

bf = Biofilter(bundle=BUNDLE, debug_mode=False) if BUNDLE else Biofilter(debug_mode=False)

_root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".biofilter.toml").is_file()),
    Path.cwd(),
)
OUTPUT_DIR = _root / "notebooks" / "templates" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

bf

### 2. Which chromosomes does this bundle actually have

Ask first. A bundle built for a subset returns `not_found` for everything
outside it — true of the bundle, not of the genome.

In [ ]:
stats = bf.report.run("platform_data_statistics", sections=["variants"]).to_pandas()

present = sorted(stats[stats["dimension_1"] == "variant_masters"]["dimension_2"].unique(),
                 key=int)
print("chromosomes in variant_masters:", present)

### 3. What the report offers

In [ ]:
print("columns:")
for column in bf.report.available_columns(REPORT):
    print(" ", column)

print("\nexample input:")
print(bf.report.example_input(REPORT))

### 4. Three input shapes, mixed freely

| shape | matches |
| --- | --- |
| `rs1225039379` | the variant that rsID maps to |
| `22:15238761` | **every** variant at that position |
| `22:20052518:C:T` | exactly that variant |

`chr`/`CHR`/`chromosome` prefixes and `:`/`-`/`_`/space separators are all
accepted; `X`, `Y`, `MT` map to 23, 24, 25.

In [ ]:
result = bf.report.run(REPORT, input_data=[
    "rs1225039379",
    "22:15238761",
    "chr22:20052518:C:T",
    "nonsense",
], most_severe_only=True)

df = result.to_pandas()
df[["input_value", "input_kind", "status", "variant_key", "rsid",
    "gene_symbol", "consequence", "note"]]

### 5. One row per transcript

A variant is annotated against every transcript it overlaps — often
dozens. The variant-level facts repeat on each row.

In [ ]:
one = bf.report.run(REPORT, input_data=["22:20052518:C:T"]).to_pandas()

print(f"{len(one)} transcripts, {one['gene_symbol'].nunique()} gene(s)")
one[["transcript_id", "consequence", "severity_rank", "impact",
     "canonical", "mane_select", "is_most_severe_for_variant"]].head(8)

`is_most_severe_for_variant` is **derived**, not stored. 4.3.0 dropped
that flag from the schema, so it is computed from
`variant_consequences.severity_rank` — consistent with whatever severity
ordering the bundle carries, rather than with what the ETL believed when
it wrote the row.

### 6. Two ways to narrow it, and they disagree

The most severe consequence is not always on the canonical transcript.

In [ ]:
for label, params in [
    ("everything", {}),
    ("most_severe_only", {"most_severe_only": True}),
    ("canonical_only", {"canonical_only": True}),
    ("both", {"most_severe_only": True, "canonical_only": True}),
]:
    out = bf.report.run(REPORT, input_data=["22:20052518:C:T"], **params).to_pandas()
    print(f"  {label:18s} {len(out):>3} rows")

### 7. Frequencies and predictions

In [ ]:
severe = bf.report.run(
    REPORT, input_data=["22:20052518:C:T"], most_severe_only=True
).to_pandas().iloc[0]

print(f"  {severe['variant_key']}  ({severe['rsid']})")
print(f"  af_joint  {severe['af_joint']:.3e}   ac {severe['ac_joint']:,} / an {severe['an_joint']:,}")
print(f"  CADD      {severe['cadd_phred']:.1f} phred")
print(f"  REVEL     {severe['revel_max']}")
print(f"  SIFT      {severe['sift_max']}     PolyPhen {severe['polyphen_max']}")
print(f"  phyloP    {severe['phylop']}")

⚠️ Frequencies are the gnomAD **joint** callset. The exomes and genomes
columns exist in `variant_masters` and are not surfaced here.

### 8. AlphaMissense scores one transcript

Of 89 transcripts, one carries a score. That is AlphaMissense's own
scope — it predicts on the canonical protein sequence — not a join that
failed.

(It did fail at first: AlphaMissense writes `ENST00000327374.9` and VEP
writes `ENST00000327374`, so the raw join matched nothing, silently.)

In [ ]:
scored = one[one["alphamissense_score"].notna()]

print(f"{len(scored)} of {len(one)} transcripts scored")
scored[["transcript_id", "consequence", "alphamissense_score",
        "alphamissense_classification"]]

### 9. Where rsIDs come from

`variant_masters` carries an `rsid` column and it is **entirely null** in
4.3.0 bundles. Lookups go through `variant_rsid`, which covers roughly
97% of variants.

In [ ]:
sample = bf.report.run(REPORT, input_data=["22:20052518:C:T"],
                       most_severe_only=True).to_pandas()
print("rsid from variant_rsid:", sample.iloc[0]["rsid"])

### 10. Export

In [ ]:
for path in result.write(OUTPUT_DIR / "annotate_variant.csv"):
    print(path)

### 11. The same thing on the command line

```bash
biofilter report run --report-name annotate_variant \\
    --input-file variants.txt \\
    --param most_severe_only=true \\
    --output variants.csv
```